# Aivora audit — Tasks 4 and 5

Two measurements that need the model, run in one session.

**Task 4 — baseline without the pipeline.** The same Qwen2.5-1.5B-Instruct, the
same 297 scored items, the same scorer, the same prompts and decoding, but
answering directly instead of through the tool pipeline. The pipeline scores
275/297; what the model scores unaided has never been measured on this split,
so the pipeline's contribution has never actually been isolated.

**Task 5 — the fresh calculation split.** The frozen calculation score is 100%,
and the 147 captions that produced it were committed *after* the eight frozen
failures were inspected. That figure is therefore not a clean held-out result.
`data/eval_fresh/calculation_fresh.jsonl` holds 44 items the vocabulary has
never seen, with expected answers computed in the builder rather than by the
pipeline.

Nothing is tuned for either. Settings: Accelerator → GPU T4, Internet → On.


In [ ]:
import subprocess, os, json, time

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"
BRANCH = "audit/nivora-v1.1"

if not os.path.exists(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, REPO_DIR], capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-600:])
    if r.returncode != 0:
        raise RuntimeError("STATUS = BLOCKED: clone failed")
os.chdir(REPO_DIR)
import sys
sys.path.insert(0, REPO_DIR)

head = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("repo at", head, "on", BRANCH)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("STATUS = BLOCKED: no GPU")
print(torch.cuda.get_device_properties(0).name)

for path in ("data/eval_fresh/calculation_fresh.jsonl",
             "scripts/eval_base_models.py",
             "scripts/run_tool_pipeline_eval.py"):
    if not os.path.exists(path):
        raise RuntimeError(f"STATUS = BLOCKED: {path} missing")
print("inputs present")


## Integrity: the frozen set must be the frozen set

Task 4 scores the frozen split, so its content hash is checked before the model
is loaded.

In [ ]:
from scripts.eval_heldout import verify_split, FROZEN
digest, status = verify_split(FROZEN, "frozen")
print(f"frozen: {status}, content_sha256 {digest[:16]}...")

import json
fresh = [json.loads(l) for l in open("data/eval_fresh/calculation_fresh.jsonl")
         if l.strip()]
print(f"fresh split: {len(fresh)} items (no manifest - not a frozen split)")


## Task 4 — the 1.5B model alone, on the frozen split

`scripts/eval_base_models.py` is the Phase 3 harness: one system prompt, greedy
decoding, the shared scorer. Using it unchanged is what makes this comparable to
the pipeline run by construction rather than by assertion.

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

started = time.time()
proc = subprocess.run(["python", "scripts/eval_base_models.py",
                       "--model", MODEL, "--shots", "0",
                       "--split", "frozen", "--force"],
                      capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("FAILED:", proc.stderr[-2000:])
    task4 = None
else:
    slug = MODEL.replace("/", "_") + "_0shot_frozen"
    with open(f"reports/heldout/{slug}.json") as handle:
        task4 = json.load(handle)
    print(f"  {time.time() - started:.0f}s")


## Task 5 — the pipeline on the fresh split

In [ ]:
started = time.time()
proc = subprocess.run(["python", "scripts/run_tool_pipeline_eval.py",
                       "--model", MODEL,
                       "--items-file", "data/eval_fresh/calculation_fresh.jsonl",
                       "--tag", "fresh", "--force"],
                      capture_output=True, text=True)
print(proc.stdout[-3000:])
if proc.returncode != 0:
    print("FAILED:", proc.stderr[-2000:])
    task5 = None
else:
    import glob
    hits = glob.glob("reports/heldout/*fresh*.json")
    with open(sorted(hits)[-1]) as handle:
        task5 = json.load(handle)
    print(f"  {time.time() - started:.0f}s")


## Results

In [ ]:
out = {"repo_commit": head, "branch": BRANCH, "model": MODEL}

print("=" * 72)
print("TASK 4 - the model WITHOUT the pipeline, frozen split")
print("=" * 72)
if task4:
    s = task4["summary"]
    out["task4_baseline_no_pipeline"] = s
    print(f"{'gate':<14}{'k/n':>10}{'acc':>9}{'95% CI':>20}")
    for gate, st in s["by_gate"].items():
        print(f"{gate:<14}{st['k']:>4}/{st['n']:<5}{st['accuracy_pct']:>8.1f}%"
              f"   [{st['ci95'][0]:.1f}, {st['ci95'][1]:.1f}]")
    o = s["overall"]
    print(f"{'OVERALL':<14}{o['k']:>4}/{o['n']:<5}{o['accuracy_pct']:>8.2f}%"
          f"   [{o['ci95'][0]:.1f}, {o['ci95'][1]:.1f}]")
    print(f"invented values {s['hallucination_invented_values']['k']}/"
          f"{s['hallucination_invented_values']['n']} "
          f"({s['hallucination_invented_values']['pct']}%)")
    print()
    print("against the pipeline on the same items: 275/297 = 92.59%, "
          "invented 0/261")
    delta = o["accuracy_pct"] - 92.59
    print(f"pipeline advantage: {-delta:+.2f} percentage points")
else:
    print("FAILED")

print()
print("=" * 72)
print("TASK 5 - the pipeline on the FRESH calculation split")
print("=" * 72)
if task5:
    s = task5["summary"]
    out["task5_fresh_split"] = s
    o = s["overall"]
    print(f"fresh calculation: {o['k']}/{o['n']} = {o['accuracy_pct']:.1f}%"
          f"   [{o['ci95'][0]:.1f}, {o['ci95'][1]:.1f}]")
    print(f"frozen calculation (vocabulary written after seeing its failures): "
          f"48/48 = 100.0%")
    print(f"span rule: {s['span_audit']['span_validated_pct']}% of "
          f"{s['span_audit']['answers_checked']}, "
          f"{s['span_audit']['violation_count']} violations")
    print(f"abstained: {s['abstained_pct']}%")
    gap = 100.0 - o["accuracy_pct"]
    print()
    print(f"gap between the contaminated and clean measurements: {gap:.1f} points")
    if gap > 10:
        print("READING: the frozen 100% substantially overstates this capability.")
    elif gap > 3:
        print("READING: the frozen 100% modestly overstates this capability.")
    else:
        print("READING: the capability holds on captions the vocabulary has "
              "never seen.")
else:
    print("FAILED")

with open("/kaggle/working/audit_gpu_results.json", "w") as handle:
    json.dump(out, handle, indent=2)
print("\nwrote /kaggle/working/audit_gpu_results.json")


## Copy the raw outputs out

Task 4 requires raw model outputs saved. Both per-item files go to the working
directory.

In [ ]:
import shutil, glob
for src in glob.glob("reports/heldout/*frozen*.json") + \
           glob.glob("reports/heldout/*fresh*.json"):
    if "Qwen" in src:
        dst = "/kaggle/working/" + os.path.basename(src)
        shutil.copy(src, dst)
        print(f"{os.path.getsize(dst)/1024:>8.0f} KB  {dst}")

# Task 4 also wants the raw outputs as JSONL.
slug = MODEL.replace("/", "_") + "_0shot_frozen"
path = f"reports/heldout/{slug}.json"
if os.path.exists(path):
    with open(path) as handle:
        recs = json.load(handle)["records"]
    with open("/kaggle/working/04_baseline_raw.jsonl", "w") as handle:
        for r in recs:
            handle.write(json.dumps(r) + "\n")
    print(f"wrote /kaggle/working/04_baseline_raw.jsonl ({len(recs)} records)")
